# PPO Sweep — Shard Runner

Runs a subset of the 35 PPO cells (7 presets x 5 seeds) at 100k timesteps.
Open 5 copies of this notebook in parallel, each with a different SHARD_ID.

**Setup**: Runtime -> Change runtime type -> A100 GPU (or T4 if A100 unavailable).

**Resilience design** (after browser/idle disconnect bug observed 2026-05-15):
- Results + state written DIRECTLY to Drive (`/content/drive/.../sweep_run/`).
  Browser disconnect or VM death no longer loses work.
- Heartbeat output every 60s in sweep.runner (added 2026-05-15) so Colab UI
  sees activity during 1-2h silent training/data-gen phases.
- Stdout via `python -u` for line-buffered output.
- Tail of training log mirrored to Drive every cell completion.

In [ ]:
# === CONFIG: change SHARD_ID for each parallel session ===
SHARD_ID = 0       # 0, 1, 2, 3, or 4
TOTAL_SHARDS = 5
TOTAL_TIMESTEPS = 100_000
LIMIT_TIME_S = 72000  # 20h safety margin within 24h Pro+ window
print(f'Shard {SHARD_ID}/{TOTAL_SHARDS}, timesteps={TOTAL_TIMESTEPS}, limit={LIMIT_TIME_S}s')

In [ ]:
# Cell 1: Mount Drive + unzip bundle
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/arcgis-farmland-mpc'
BUNDLE = f'{DRIVE_ROOT}/ppo_sweep_bundle.zip'
assert os.path.exists(BUNDLE), f'Upload ppo_sweep_bundle.zip to {DRIVE_ROOT}'
!unzip -q -o $BUNDLE -d /content
!ls /content/src

In [ ]:
# Cell 2: Install deps
!pip install -q sb3-contrib geopandas libpysal opensimplex rasterio pyyaml shapely

In [ ]:
# Cell 3: Env vars + GPU check
import os, subprocess
os.environ['TEST_SRC_ROOT'] = '/content/src/test'
os.environ['ADK_SRC_ROOT']  = '/content/src/adk'
# Force unbuffered Python so heartbeat lines reach Colab UI immediately.
os.environ['PYTHONUNBUFFERED'] = '1'
print(subprocess.check_output(['nvidia-smi'], text=True)[:500])

In [ ]:
# Cell 4: Generate all 35 datasets (shared across shards, ~30 min)
# Datasets land on local /content (not Drive) -- they're cheap to regenerate
# and large in IO. Resume across sessions skips already-generated.
%cd /content/src/benchmark
import pathlib
presets = ['bishan_clone','neijiang_clone','plain_small_cons','plain_large_cons',
           'plain_medium_frag','mixed_medium_frag','hilly_small_cons']
for p in presets:
    for s in range(5):
        out = f'data_dev/{p}_seed{s}'
        if pathlib.Path(f'{out}/manifest.json').exists():
            print(f'skip {out}')
            continue
        !python -u -m generator.generate --preset presets/{p}.yaml --seed {s} --out {out}
print('Done: datasets generated')

In [ ]:
# Cell 5: Run PPO sweep for this shard.
# Results + state go DIRECTLY to Drive so browser disconnect can't lose work.
import os, pathlib
RUN_DIR = f'{DRIVE_ROOT}/sweep_run'
RESULTS_ROOT = f'{RUN_DIR}/results'   # all 5 shards write here, share preset/method/seed dirs
STATE_PATH   = f'{RUN_DIR}/sweep_state_ppo_shard{SHARD_ID}.json'
LOG_PATH     = f'{RUN_DIR}/sweep_ppo_shard{SHARD_ID}.log'
pathlib.Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
pathlib.Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)
%cd /content/src/benchmark
# 2>&1 | tee duplicates output to a Drive-side log so even if the browser
# disconnects, the log file on Drive grows and we can resume tracking.
!python -u -m sweep.runner \
  --manifest sweep_manifest.csv \
  --state "$STATE_PATH" \
  --data-root data_dev \
  --results-root "$RESULTS_ROOT" \
  --only-method PPO \
  --shard {SHARD_ID}/{TOTAL_SHARDS} \
  --resume \
  --limit-time {LIMIT_TIME_S} 2>&1 | tee -a "$LOG_PATH"

In [ ]:
# Cell 6: Sanity report (results already on Drive)
import json, pathlib
from collections import Counter
state = json.loads(pathlib.Path(STATE_PATH).read_text())
ppo = [c for c in state['cells'] if c['method'] == 'PPO']
shard_cells = [c for i, c in enumerate(ppo) if i % TOTAL_SHARDS == SHARD_ID]
print(f'Shard {SHARD_ID}: {Counter(c["status"] for c in shard_cells)}')
for c in shard_cells:
    if c['status'] == 'done':
        print(f"  OK    {c['cell_id']}")
    elif c['status'] == 'failed':
        print(f"  FAIL  {c['cell_id']}: {(c.get('error') or '')[:120]}")
print(f'\nResults dir: {RESULTS_ROOT}')
print(f'State:       {STATE_PATH}')
print(f'Log:         {LOG_PATH}')